<a href="https://colab.research.google.com/github/gokulbytes/mind-kg-news-recommender/blob/main/recommender.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Requirements

In [1]:
# Install necessary libraries
!pip install pandas numpy networkx -q

In [2]:
# Import necessary libraries
import pandas as pd
import numpy as np
import networkx as nx
import json
import ast
from tqdm import tqdm

import warnings
warnings.filterwarnings('ignore')

# main

## behaviors_df

In [3]:
# Load the dataset
behaviors_df = pd.read_csv("/content/behaviors.tsv", sep="\t", header=None, names=['ImpressionID', 'UserID', 'TimeStamp', 'History', 'Impressions'])
behaviors_df.head()

,ImpressionID,UserID,TimeStamp,History,Impressions
0,1,U13740,11/11/2019 9:05:58 AM,N55189 N42782 N34694 N45794 N18445 N63302 N104...,N55689-1 N35729-0
1,2,U91836,11/12/2019 6:11:30 PM,N31739 N6072 N63045 N23979 N35656 N43353 N8129...,N20678-0 N39317-0 N58114-0 N20495-0 N42977-0 N...
2,3,U73700,11/14/2019 7:01:48 AM,N10732 N25792 N7563 N21087 N41087 N5445 N60384...,N50014-0 N23877-0 N35389-0 N49712-0 N16844-0 N...
3,4,U34670,11/11/2019 5:28:05 AM,N45729 N2203 N871 N53880 N41375 N43142 N33013 ...,N35729-0 N33632-0 N49685-1 N27581-0
4,5,U8125,11/12/2019 4:11:21 PM,N10078 N56514 N14904 N33740,N39985-0 N36050-0 N16096-0 N8400-1 N22407-0 N6...


In [4]:
behaviors_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 156965 entries, 0 to 156964
Data columns (total 5 columns):
 #   Column        Non-Null Count   Dtype 
---  ------        --------------   ----- 
 0   ImpressionID  156965 non-null  int64 
 1   UserID        156965 non-null  object
 2   TimeStamp     156965 non-null  object
 3   History       153727 non-null  object
 4   Impressions   156965 non-null  object
dtypes: int64(1), object(4)
memory usage: 6.0+ MB


### Drop Duplicates

In [5]:
# Drop duplicate rows and reset the index
behaviors_df = behaviors_df.drop_duplicates().reset_index(drop=True)

## news_df

In [6]:
# Load the dataset
news_df = pd.read_csv("/content/news.tsv", sep="\t", header=None, names=['NewsID', 'Category', 'SubCategory', 'Title', 'Abstract', 'URL','TitleEntities','AbstractEntities'])
news_df.head()

,NewsID,Category,SubCategory,Title,Abstract,URL,TitleEntities,AbstractEntities
0,N55528,lifestyle,lifestyleroyals,"The Brands Queen Elizabeth, Prince Charles, an...","Shop the notebooks, jackets, and more that the...",https://assets.msn.com/labs/mind/AAGH0ET.html,"[{""Label"": ""Prince Philip, Duke of Edinburgh"",...",[]
1,N19639,health,weightloss,50 Worst Habits For Belly Fat,These seemingly harmless habits are holding yo...,https://assets.msn.com/labs/mind/AAB19MK.html,"[{""Label"": ""Adipose tissue"", ""Type"": ""C"", ""Wik...","[{""Label"": ""Adipose tissue"", ""Type"": ""C"", ""Wik..."
2,N61837,news,newsworld,The Cost of Trump's Aid Freeze in the Trenches...,Lt. Ivan Molchanets peeked over a parapet of s...,https://assets.msn.com/labs/mind/AAJgNsz.html,[],"[{""Label"": ""Ukraine"", ""Type"": ""G"", ""WikidataId..."
3,N53526,health,voices,I Was An NBA Wife. Here's How It Affected My M...,"I felt like I was a fraud, and being an NBA wi...",https://assets.msn.com/labs/mind/AACk2N6.html,[],"[{""Label"": ""National Basketball Association"", ..."
4,N38324,health,medical,"How to Get Rid of Skin Tags, According to a De...","They seem harmless, but there's a very good re...",https://assets.msn.com/labs/mind/AAAKEkt.html,"[{""Label"": ""Skin tag"", ""Type"": ""C"", ""WikidataI...","[{""Label"": ""Skin tag"", ""Type"": ""C"", ""WikidataI..."


In [7]:
news_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 51282 entries, 0 to 51281
Data columns (total 8 columns):
 #   Column            Non-Null Count  Dtype 
---  ------            --------------  ----- 
 0   NewsID            51282 non-null  object
 1   Category          51282 non-null  object
 2   SubCategory       51282 non-null  object
 3   Title             51282 non-null  object
 4   Abstract          48616 non-null  object
 5   URL               51282 non-null  object
 6   TitleEntities     51279 non-null  object
 7   AbstractEntities  51278 non-null  object
dtypes: object(8)
memory usage: 3.1+ MB


### Drop Duplicates

In [8]:
# Drop duplicate rows and reset the index
news_df = news_df.drop_duplicates().reset_index(drop=True)

## Data Preprocesssing

### behaviors_df

In [9]:
# Fill missing values in the 'History' column with empty strings
behaviors_df['History'].fillna("", inplace=True)

behaviors_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 156965 entries, 0 to 156964
Data columns (total 5 columns):
 #   Column        Non-Null Count   Dtype 
---  ------        --------------   ----- 
 0   ImpressionID  156965 non-null  int64 
 1   UserID        156965 non-null  object
 2   TimeStamp     156965 non-null  object
 3   History       156965 non-null  object
 4   Impressions   156965 non-null  object
dtypes: int64(1), object(4)
memory usage: 6.0+ MB


### news_df

In [10]:
# Fill missing values in the 'Abstract' column with empty strings
news_df['Abstract'].fillna("", inplace=True)

# Fill missing values in the 'TitleEntities' column with empty strings
news_df['TitleEntities'].fillna("[]", inplace=True)

# Fill missing values in the 'AbstractEntities' column with empty strings
news_df['AbstractEntities'].fillna("[]", inplace=True)

In [11]:
news_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 51282 entries, 0 to 51281
Data columns (total 8 columns):
 #   Column            Non-Null Count  Dtype 
---  ------            --------------  ----- 
 0   NewsID            51282 non-null  object
 1   Category          51282 non-null  object
 2   SubCategory       51282 non-null  object
 3   Title             51282 non-null  object
 4   Abstract          51282 non-null  object
 5   URL               51282 non-null  object
 6   TitleEntities     51282 non-null  object
 7   AbstractEntities  51282 non-null  object
dtypes: object(8)
memory usage: 3.1+ MB


## Knowledge Graph

In [12]:
def load_embeddings(file_path):
    embeddings = {}
    with open(file_path, "r") as file:
        for line in file:
            parts = line.strip().split("\t")
            entities = parts[0]
            vector = np.array([float(x) for x in parts[1:]])
            embeddings[entities] = vector

    return embeddings

# Load entity embeddings
entity_embeddings = load_embeddings("/content/entity_embedding.vec")

# Load relation embeddings
relation_embeddings = load_embeddings("/content/relation_embedding.vec")

In [13]:
def build_knowledge_graph(users, news):
    graph = nx.Graph()

    # Add News Nodes and Content Edges
    for i, row in tqdm(news.iterrows(), total=len(news), desc="Adding News Nodes"):
        news_id = row['NewsID']
        graph.add_node(news_id, type="news", category=row['Category'])

        # Link News to Category
        graph.add_node(row['Category'], type="category")
        graph.add_edge(news_id, row['Category'], label="belongs_to")

        # Parse and Link Entities
        for field in ['TitleEntities', 'AbstractEntities']:
            try:
                entities = ast.literal_eval(row[field])
                for entity in entities:
                    entity_id = entity['WikidataId']
                    graph.add_node(entity_id, type="entity")
                    graph.add_edge(news_id, entity_id, label="mentions")
            except:
                continue

    # Add User Nodes and History Edges
    for i, row in tqdm(users.iterrows(), total=len(users), desc="Adding User Edges"):
        user_id = row['UserID']
        history = row['History'].split()
        graph.add_node(user_id, type="user")
        for news_id in history:
            if graph.has_node(news_id):
                graph.add_edge(user_id, news_id, label="clicked")

    return graph

knowledge_graph = build_knowledge_graph(behaviors_df, news_df)

Adding User Edges: 100%|██████████| 156965/156965 [00:20<00:00, 7741.03it/s]


## Recommendation Function

In [14]:
def get_recommendations(user_id, graph, n=10):
    if user_id not in graph:
        return "User not found."

    # Get User History
    user_entities = []
    clicked_news = [i for i in graph.neighbors(user_id) if graph.nodes[i].get("type") == "news"]
    for news_id in clicked_news:
        user_entities.extend([entity for entity in graph.neighbors(news_id) if graph.nodes[entity].get("type") == "entity"])

    # Verctorizing Users
    user_vecs = [entity_embeddings[entity] for entity in user_entities if entity in entity_embeddings]
    if not user_vecs:
        return "No entity data for user."
    user_profile = np.mean(user_vecs, axis=0)

    # Generate Candidate News
    candidates = {}
    for news_id in clicked_news:
        category = graph.nodes[news_id].get("category")
        for neighbor_news in graph.neighbors(category):
            if neighbor_news not in clicked_news and graph.nodes[neighbor_news].get("type") == "news":
                # Calculate Similarity Score
                news_entities = [entity for entity in graph.neighbors(neighbor_news) if graph.nodes[entity].get("type") == "entity"]
                news_vecs = [entity_embeddings[entity] for entity in news_entities if entity in entity_embeddings]

                if news_vecs:
                    news_profile = np.mean(news_vecs, axis=0)
                    score = np.dot(user_profile, news_profile) / (np.linalg.norm(user_profile) * np.linalg.norm(news_profile))
                    candidates[neighbor_news] = score

    sorted_news = sorted(candidates.items(), key=lambda x: x[1], reverse=True)[:n]
    return sorted_news

### Recommendations

In [17]:
# Select a sample user
sample_user = behaviors_df['UserID'].iloc[0]
recommendations = get_recommendations(sample_user, knowledge_graph)

print(f"Top 10 Recommendations for User: {sample_user}")
for i, (news_id, score) in enumerate(recommendations, 1):
    title = news_df[news_df['NewsID'] == news_id]['Title'].values[0]
    print(f"{i}. [{news_id}] Score: {score:.4f} | {title}")

Top 10 Recommendations for User: U61194
1. [N54283] Score: 0.8719 | Prince Harry just hinted that he and Meghan Markle may want another baby
2. [N22288] Score: 0.8372 | Chrissy Teigen and John Legend's 2020 pick, their framed Trump tweet and more takeaways from their Vanity Fair interview
3. [N58553] Score: 0.8301 | Prince Harry and Meghan Markle Will Reportedly Visit Oprah While in California
4. [N8073] Score: 0.8240 | Jane Fonda, Rosanna Arquette, Catherine Keener arrested at climate change protest: 'Women bear the brunt'
5. [N38787] Score: 0.8187 | Giuliani needed Apple genius help to unlock his iPhone after named Trump cybersecurity adviser
6. [N5104] Score: 0.8169 | Oprah Winfrey Just Bought Jeff Bridges' Ranch in Montecito
7. [N61203] Score: 0.8083 | Zendaya, Nicole Kidman and More Best Dressed Stars at Elle's Women in Hollywood Event
8. [N21843] Score: 0.8060 | Reeker expected to testify this weekend in impeachment inquiry
9. [N24202] Score: 0.8045 | Jennifer Aniston, Mariah Car

## Save Models

In [18]:
import joblib

# Save entity_embeddings
joblib.dump(entity_embeddings, "entity_embeddings.joblib")
print("Entity embeddings saved to entity_embeddings.joblib")

# Save news_metadata
joblib.dump(news_df[['NewsID', 'Title', 'Category']], "news_metadata.joblib")
print("News metadata saved to news_metadata.joblib")

# Save knowledge graph
joblib.dump(knowledge_graph, "kg_recommender.joblib")
print("Knowledgs graph saved to kg_recommender.joblib")

Entity embeddings saved to entity_embeddings.joblib
News metadata saved to news_metadata.joblib
Knowledgs graph saved to kg_recommender.joblib
